# WAI Studio · tạo ảnh anime từ Google Colab bằng liên kết tạm thời

> **Không cần Cloudflare, API token hay máy chủ GPU khác.** Colab chạy checkpoint WAI-illustrious v17 + LoRA đã kiểm SHA-256; Gradio chỉ hiển thị giao diện. Ảnh mẫu web Cloudflare không liên quan tới model này.

### Chạy trong 3 bước
1. Trong Colab chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU mạnh hơn). Bấm **Runtime → Run all**. Nếu được hỏi, cho phép gắn Google Drive để lưu checkpoint/ảnh cho các phiên sau. Lần đầu cần tải ~6,94 GB model + tối đa ~457 MB LoRA và có thể mất một lúc; những lần sau dùng lại các file đã xác minh.
2. Ở ô cuối, chờ dòng `Running on public URL` rồi mở liên kết `https://....gradio.live` — **không cần tài khoản hay mật khẩu**. Dùng giao diện để tạo ảnh, ảnh → ảnh, tô mask sửa tay/chân/mắt và tải PNG.
3. **Giữ notebook Colab đang kết nối.** Link này chỉ tồn tại khi phiên Colab/Gradio còn chạy (có thể hết hạn sớm khi runtime bị ngắt). Đóng phiên bằng cách dừng runtime; lần sau chạy notebook để có link mới. Nếu dùng Drive, ảnh nằm ở `MyDrive/AI/outputs`; nếu không lưu được Drive, hãy tải từ giao diện hoặc `/content/wai_outputs` trước khi hết phiên.

**Lưu ý bảo mật:** `share=True` tạo URL *truy cập được từ Internet* qua proxy Gradio và **không có đăng nhập**. Bất kỳ ai biết URL đều có thể dùng giao diện và GPU Colab của bạn. Không chia sẻ URL hoặc lưu notebook có output chứa URL ở nơi công khai; dừng runtime để ngắt link. File checkpoint/LoRA bị chặn khỏi đường tải file Gradio, nhưng URL không phải cơ chế xác thực. Gradio nhận dữ liệu qua đường hầm TLS; CPU/GPU và checkpoint vẫn chạy trên Colab. [Giải thích share link](https://www.gradio.app/guides/understanding-gradio-share-links). Ảnh có thể chứa thông tin trong metadata nếu bạn tự bật tùy chọn này.

Chỉ bật LoRA đã chọn trước khi nạp model (ô 3–6). Trong giao diện có thể tắt/bật và đổi cường độ **các LoRA đã nạp** mà không cần tải lại. Inpainting từ checkpoint SDXL gốc định hướng vùng trắng, ghép lại để giữ pixel đen; không đảm bảo sửa đúng mọi lỗi. Colab không bảo đảm GPU liên tục hay đủ RAM/ổ đĩa. Để đổi đường dẫn, chế độ bộ nhớ hoặc danh sách LoRA, hãy **khởi động lại runtime và Run all**; không chạy lại riêng ô nạp model khi giao diện còn giữ pipeline cũ.


In [ ]:
# @title 1. Kiểm tra GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Chọn Runtime → Change runtime type → GPU, rồi chạy lại ô này.")
device = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"GPU: {device.name} | VRAM trống: {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# @title 2. Tự cài thư viện còn thiếu (giữ PyTorch/CUDA của Colab)
%pip -q install "diffusers==0.35.2" "transformers==4.52.4" "accelerate==1.10.1" "peft==0.17.1" "safetensors>=0.4.5,<1" "huggingface-hub==0.36.2" "hf-xet>=1.1.3,<2" "gradio==6.15.2" "pydantic>=2.12.5,<3" "starlette>=1.3.1,<2"

from importlib.metadata import PackageNotFoundError, version
from packaging.specifiers import SpecifierSet
import gradio, gradio_client, pydantic, starlette, huggingface_hub

versions = {
    "gradio": gradio.__version__,
    "gradio-client": gradio_client.__version__,
    "pydantic": pydantic.__version__,
    "starlette": starlette.__version__,
    "huggingface-hub": huggingface_hub.__version__,
}
for package in ("diffusers", "transformers", "accelerate", "peft", "safetensors", "hf-xet"):
    try:
        versions[package] = version(package)
    except PackageNotFoundError as exc:
        raise RuntimeError(f"Thiếu {package}. Cài đặt ô 2 chưa hoàn tất; xem lỗi pip ở phía trên.") from exc
required = {
    "gradio": "==6.15.2",
    "gradio-client": "==2.5.0",
    "pydantic": ">=2.12.5,<3",
    "starlette": ">=1.3.1,<2",
    "huggingface-hub": "==0.36.2",
    "diffusers": "==0.35.2",
    "transformers": "==4.52.4",
    "accelerate": "==1.10.1",
    "peft": "==0.17.1",
    "safetensors": ">=0.4.5,<1",
    "hf-xet": ">=1.1.3,<2",
}
for package, constraint in required.items():
    if versions[package] not in SpecifierSet(constraint):
        raise RuntimeError(f"{package} đang là {versions[package]}, cần {constraint}. Chọn Runtime → Restart runtime rồi Run all.")
# Bắt lỗi import trước khi tải checkpoint 6,94 GB ở ô 4.
try:
    from diffusers import StableDiffusionXLPipeline, AutoPipelineForImage2Image, AutoPipelineForInpainting
except Exception as exc:
    raise RuntimeError("Diffusers không import được. Chọn Runtime → Restart runtime rồi Run all; nếu vẫn lỗi, gửi traceback ô 2 (che thông tin riêng).") from exc
print("✅ Thư viện Studio đã sẵn sàng:", versions)


In [ ]:
# @title 3. Cấu hình model, LoRA, Drive và bộ nhớ { display-mode: "form" }
MOUNT_DRIVE = True # @param {type:"boolean"}
MODEL_PATH = "/content/drive/MyDrive/AI/models/WAI-illustrious.safetensors" # @param {type:"string"}
AUTO_DOWNLOAD = True # @param {type:"boolean"}
PERSIST_MODEL_TO_DRIVE = True # @param {type:"boolean"}
CACHE_MODEL_LOCAL = True # @param {type:"boolean"}
USE_ANATOMY_LORA = True # @param {type:"boolean"}
ANATOMY_WEIGHT = 0.55 # @param {type:"slider", min:0.1, max:1.0, step:0.05}
ANATOMY_LORA_PATH = "" # @param {type:"string"}
USE_EYE_LORA = True # @param {type:"boolean"}
EYE_WEIGHT = 0.45 # @param {type:"slider", min:0.1, max:1.0, step:0.05}
EYE_LORA_PATH = "" # @param {type:"string"}
PERSIST_LORAS_TO_DRIVE = True # @param {type:"boolean"}
OUTPUT_DIR = "/content/drive/MyDrive/AI/outputs" # @param {type:"string"}
VRAM_MODE = "auto" # @param ["auto", "speed", "low_vram"]

from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

source_model = Path(MODEL_PATH).expanduser()
output_dir = Path(OUTPUT_DIR).expanduser()
local_cache_root = Path("/content/wai_model_cache")
local_lora_cache = Path("/content/wai_lora_cache")
drive_root = Path("/content/drive")
lora_drive_dir = drive_root / "MyDrive" / "AI" / "loras"
if not source_model.is_absolute() or not output_dir.is_absolute():
    raise ValueError("MODEL_PATH và OUTPUT_DIR phải là đường dẫn tuyệt đối.")
if source_model.suffix.lower() != ".safetensors":
    raise ValueError("MODEL_PATH phải kết thúc bằng .safetensors (checkpoint đầy đủ, không phải LoRA).")
using_drive = any(drive_root == p or drive_root in p.parents for p in (source_model, output_dir))
if using_drive and not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive chưa được gắn. Bật MOUNT_DRIVE rồi chạy lại ô 3, hoặc đổi đường dẫn sang /content.")
if VRAM_MODE not in ("auto", "speed", "low_vram"):
    raise ValueError("VRAM_MODE phải là auto, speed hoặc low_vram.")
for weight in (ANATOMY_WEIGHT, EYE_WEIGHT):
    if not 0.1 <= weight <= 1.0:
        raise ValueError("Cường độ LoRA phải trong khoảng 0.1–1.0.")
print("Model có sẵn:" if source_model.is_file() else "Model chưa có (sẽ tự tải ở ô 4):", source_model)
print("LoRA bật:", ", ".join(name for enabled, name in ((USE_ANATOMY_LORA, "anatomy"), (USE_EYE_LORA, "eyes")) if enabled) or "không")
print("Thư mục lưu ảnh:", output_dir)

In [ ]:
# @title 4. Tự tải/checkpoint: ưu tiên file có sẵn, kiểm tra SHA-256 khi tải mới
import hashlib
import os
import shutil
# Tăng thời gian chờ trên mạng Colab chập chờn; đặt trước khi import huggingface_hub.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
from huggingface_hub import hf_hub_download
from safetensors import safe_open

# Bản v17 pruned FP16; hash trùng file Civitai model version 2883731.
HF_REPO = "LyliaEngine/waiIllustriousSDXL_v170"
HF_FILENAME = "waiIllustriousSDXL_v170.safetensors"
HF_REVISION = "32be7bfdcd406db70df663b9cee3313957deb68f"
HF_SHA256 = "f116b0c78ff441467b0cdc8f1936e1ed18ea31e9997c7b132b1b8db533f0bd04"
HF_MODEL_BYTES = 6_938_040_682
MIN_CHECKPOINT_BYTES = 100 * 2**20
DISK_RESERVE_BYTES = 2 * 2**30

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 2**20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def inspect_checkpoint(path, verify_official=False):
    path = Path(path)
    if path.suffix.lower() != ".safetensors" or not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy checkpoint .safetensors: {path}")
    size = path.stat().st_size
    if size < MIN_CHECKPOINT_BYTES:
        raise ValueError("File model quá nhỏ: có thể tải dở, là HTML hoặc là LoRA.")
    if verify_official:
        if size != HF_MODEL_BYTES or sha256_file(path) != HF_SHA256:
            raise ValueError("SHA-256/dung lượng model tải về không trùng bản v17 gốc; không nạp file này.")
    try:
        with safe_open(str(path), framework="pt", device="cpu") as header:
            keys = header.keys()  # chỉ đọc header, không tải trọng số vào RAM
            has_unet = any(key.startswith("model.diffusion_model.") for key in keys)
            has_clip = any(key.startswith("conditioner.embedders.") for key in keys)
    except Exception as exc:
        raise ValueError("Không đọc được safetensors; hãy kiểm tra file checkpoint.") from exc
    if not (has_unet and has_clip):
        raise ValueError("Cần checkpoint SDXL đầy đủ (UNet + text encoder), không phải LoRA/UNet-only.")
    return size

def copy_atomic(source, destination, size, preserve_mtime=False):
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".partial")
    try:
        # Drive FUSE có thể không cho phép copy2/copystat; chỉ giữ mtime khi cache vào /content.
        if preserve_mtime:
            shutil.copy2(source, partial)
        else:
            shutil.copyfile(source, partial)
        if partial.stat().st_size != size:
            raise OSError("Bản sao checkpoint không đầy đủ.")
        os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)

WAI_STUDIO_VERSION_VERIFIED = False
if source_model.exists() and not source_model.is_file():
    raise ValueError(f"MODEL_PATH là thư mục hoặc tệp đặc biệt, không phải checkpoint: {source_model}")
if source_model.is_file():
    model_size = inspect_checkpoint(source_model)  # file của người dùng có thể là phiên bản khác
    checkpoint = source_model
    if CACHE_MODEL_LOCAL and drive_root in source_model.parents:
        local_cache_root.mkdir(parents=True, exist_ok=True)
        cached = local_cache_root / source_model.name
        same_file = (
            cached.is_file()
            and cached.stat().st_size == model_size
            and cached.stat().st_mtime_ns == source_model.stat().st_mtime_ns
        )
        if same_file:
            checkpoint = cached
            print("Dùng lại bản sao /content trong phiên này.")
        elif shutil.disk_usage(local_cache_root).free >= model_size + DISK_RESERVE_BYTES:
            try:
                print("Sao chép model từ Drive sang /content để nạp nhanh hơn...")
                copy_atomic(source_model, cached, model_size, preserve_mtime=True)
                checkpoint = cached
            except OSError as exc:
                print(f"Không sao chép được ({type(exc).__name__}); nạp trực tiếp từ Drive.")
        else:
            print("Đĩa /content không đủ chỗ để sao chép; nạp trực tiếp từ Drive.")
    # Băm đúng file SẼ nạp, kể cả bản sao/cache /content (có thể khác nguồn Drive).
    if model_size == HF_MODEL_BYTES:
        checkpoint_hash = sha256_file(checkpoint)
        if checkpoint_hash != HF_SHA256 and checkpoint != source_model:
            if sha256_file(source_model) == HF_SHA256:
                print("Cache /content sai SHA-256; bỏ qua bản sao và nạp bản gốc trên Drive.")
                checkpoint = source_model
                checkpoint_hash = HF_SHA256
        if checkpoint_hash == HF_SHA256:
            WAI_STUDIO_VERSION_VERIFIED = True
            print("Checkpoint có sẵn: đã xác minh là WAI-illustrious v17 (SHA-256 trùng Civitai).")
        else:
            print("Checkpoint có sẵn: chỉ kiểm tra định dạng SDXL, KHÔNG xác thực là WAI v17; LoRA chỉ hợp với họ Illustrious.")
    else:
        print("Checkpoint có sẵn: chỉ kiểm tra định dạng SDXL, KHÔNG xác thực là WAI v17; LoRA chỉ hợp với họ Illustrious.")
else:
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f"Chưa có model: {source_model}. Bật AUTO_DOWNLOAD ở ô 3 hoặc đặt checkpoint vào đúng đường dẫn.")
    local_cache_root.mkdir(parents=True, exist_ok=True)
    disk_ok = shutil.disk_usage(local_cache_root).free >= HF_MODEL_BYTES + DISK_RESERVE_BYTES
    on_drive = drive_root in source_model.parents
    print(f"Chưa có checkpoint; đang tải bản WAI-illustrious v17 ({HF_MODEL_BYTES / 10**9:.2f} GB)...")
    if disk_ok:
        try:
            downloaded = Path(hf_hub_download(
                repo_id=HF_REPO, filename=HF_FILENAME, revision=HF_REVISION,
                cache_dir=str(local_cache_root), token=False,
            ))
        except Exception as exc:
            raise RuntimeError("Không tải được model từ Hugging Face. Kiểm tra Internet hoặc tự đặt file ở MODEL_PATH.") from exc
        inspect_checkpoint(downloaded, verify_official=True)
        WAI_STUDIO_VERSION_VERIFIED = True
        checkpoint = downloaded  # không sao chép lại sang /content lần thứ hai
        if on_drive and PERSIST_MODEL_TO_DRIVE:
            try:
                print("Lưu bản đã xác minh vào Drive để phiên sau không tải lại...")
                copy_atomic(downloaded, source_model, HF_MODEL_BYTES)
            except OSError as exc:
                print(f"Drive không lưu được ({type(exc).__name__}); vẫn tạo ảnh từ bản tạm ở /content.")
    elif on_drive and PERSIST_MODEL_TO_DRIVE:
        # Đĩa máy Colab quá ít: tải vào Drive, không tạo bản sao 7 GB trên /content.
        source_model.parent.mkdir(parents=True, exist_ok=True)
        try:
            downloaded = Path(hf_hub_download(
                repo_id=HF_REPO, filename=HF_FILENAME, revision=HF_REVISION,
                local_dir=str(source_model.parent), token=False,
            ))
            inspect_checkpoint(downloaded, verify_official=True)
            WAI_STUDIO_VERSION_VERIFIED = True
            if downloaded != source_model:
                os.replace(downloaded, source_model)
            checkpoint = source_model
        except Exception as exc:
            raise RuntimeError("Không đủ đĩa /content và tải vào Drive không thành công. Kiểm tra dung lượng/quyền Drive.") from exc
    else:
        raise OSError("Thiếu đĩa trống để tải model (~9 GiB). Giải phóng đĩa, hoặc bật lưu vào Drive ở ô 3.")
    print("Đã đối chiếu SHA-256 với bản v17 gốc.")
print(f"Sẵn sàng: {checkpoint} ({checkpoint.stat().st_size / 2**30:.2f} GiB)")
if not WAI_STUDIO_VERSION_VERIFIED:
    raise RuntimeError("Studio chỉ nhận checkpoint WAI-illustrious v17 đúng SHA-256. Đổi MODEL_PATH sang bản gốc, rồi chạy lại ô 4.")


In [ ]:
# @title 5. Tự tải và xác minh LoRA Illustrious cho tay/chân/mắt (có thể tắt ở ô 3)
# HF là bản lưu của bên thứ ba; SHA-256 đầy đủ trùng file phát hành Civitai ở các version ID sau.
ANATOMY_LORA_REPO = "ench100/bodyandface"
ANATOMY_LORA_FILE = "anatomy_helper.safetensors"
ANATOMY_LORA_REV = "bed49d45df95c0695aedad3b2aa6aff389fb3777"
ANATOMY_LORA_SHA256 = "bf6a950036b7599212a2c68d65f3ba07b28689067e167915d2a0ecb2018c26ca"
ANATOMY_LORA_BYTES = 228_473_940  # Civitai version 1318504, file 1223247
EYE_LORA_REPO = "Muapi/eyes-for-illustrious-lora-perfect-anime-eyes"
EYE_LORA_FILE = "eyes-for-illustrious-lora-perfect-anime-eyes.safetensors"
EYE_LORA_REV = "1abbc862f53f5101962ebf1c337513aff91bd206"
EYE_LORA_SHA256 = "97c1a083ffe6b4d45c545196eabd01c754936b996ade0c9db6d072f3bd340c55"
EYE_LORA_BYTES = 228_457_660  # Civitai version 2066663, file 1963176
LORA_DISK_RESERVE_BYTES = 256 * 2**20
if "checkpoint" not in globals():
    raise RuntimeError("Hãy chạy ô 4 để chuẩn bị checkpoint trước khi tải LoRA.")

lora_manifest = {
    "anatomy": dict(enabled=USE_ANATOMY_LORA, weight=ANATOMY_WEIGHT, manual=ANATOMY_LORA_PATH,
                    repo=ANATOMY_LORA_REPO, filename=ANATOMY_LORA_FILE, revision=ANATOMY_LORA_REV,
                    sha256=ANATOMY_LORA_SHA256, size=ANATOMY_LORA_BYTES, version="Civitai:1318504"),
    "eyes": dict(enabled=USE_EYE_LORA, weight=EYE_WEIGHT, manual=EYE_LORA_PATH,
                 repo=EYE_LORA_REPO, filename=EYE_LORA_FILE, revision=EYE_LORA_REV,
                 sha256=EYE_LORA_SHA256, size=EYE_LORA_BYTES, version="Civitai:2066663"),
}

def inspect_lora(path, spec):
    path = Path(path)
    if path.suffix.lower() != ".safetensors" or not path.is_file():
        raise FileNotFoundError(f"LoRA cần là file .safetensors: {path}")
    if path.stat().st_size != spec["size"] or sha256_file(path) != spec["sha256"]:
        raise ValueError(f"LoRA {spec['version']} không đúng kích thước/SHA-256: {path}. Không nạp file sai phiên bản.")
    try:
        with safe_open(str(path), framework="pt", device="cpu") as header:
            if not any(key.startswith("lora_unet_") or key.startswith("unet.") for key in header.keys()):
                raise ValueError("Không có lớp LoRA UNet kiểu SDXL/Kohya.")
    except Exception as exc:
        raise ValueError(f"Không đọc được header LoRA: {path}") from exc
    return path

def prepare_lora(spec):
    manual = spec["manual"].strip()
    if manual:
        return inspect_lora(Path(manual).expanduser(), spec)  # chấp nhận tên file Civitai nếu SHA trùng
    have_drive = (drive_root / "MyDrive").is_dir()
    saved = lora_drive_dir / spec["filename"]
    if have_drive and saved.is_file():
        print(f"Dùng lại bản LoRA đã xác minh trên Drive: {saved}")
        return inspect_lora(saved, spec)
    local_lora_cache.mkdir(parents=True, exist_ok=True)
    disk_ok = shutil.disk_usage(local_lora_cache).free >= spec["size"] + LORA_DISK_RESERVE_BYTES
    if not disk_ok and not (have_drive and PERSIST_LORAS_TO_DRIVE):
        raise OSError("Thiếu đĩa /content cho LoRA. Giải phóng ~0.5 GiB, hoặc gắn Drive và bật PERSIST_LORAS_TO_DRIVE.")
    print(f"Tải LoRA {spec['version']} từ Hugging Face, commit {spec['revision'][:8]}...")
    try:
        args = dict(repo_id=spec["repo"], filename=spec["filename"],
                    revision=spec["revision"], token=False)
        if disk_ok:
            args["cache_dir"] = str(local_lora_cache)  # không tạo thêm một bản trong /content
        else:
            lora_drive_dir.mkdir(parents=True, exist_ok=True)
            args["local_dir"] = str(lora_drive_dir)  # ít đĩa: tải thẳng vào Drive
        downloaded = inspect_lora(Path(hf_hub_download(**args)), spec)
    except Exception as exc:
        raise RuntimeError(f"LoRA {spec['version']} tải/xác minh thất bại. Dùng file gốc tại ANATOMY_LORA_PATH/EYE_LORA_PATH, hoặc tắt LoRA này ở ô 3.") from exc
    if disk_ok and have_drive and PERSIST_LORAS_TO_DRIVE:
        try:
            copy_atomic(downloaded, saved, spec["size"])
            try:
                inspect_lora(saved, spec)  # xác minh cả bản trên Drive, không chỉ kiểm tra size
            except ValueError:
                saved.unlink(missing_ok=True)  # chỉ xóa bản vừa do notebook sao chép
                raise
            print(f"Đã lưu bản sao xác minh trên Drive: {saved}")
        except (OSError, ValueError) as exc:
            print(f"Không lưu được LoRA lên Drive ({type(exc).__name__}); vẫn dùng bản đã xác minh trong /content.")
    return downloaded

lora_paths = {}
for name, spec in lora_manifest.items():
    if spec["enabled"]:
        lora_paths[name] = prepare_lora(spec)
        print(f"Đã xác minh {name}: {spec['version']} | SHA-256 {spec['sha256']}")
if not lora_paths:
    print("Không bật LoRA; ảnh vẫn dùng checkpoint gốc. Bật ở ô 3 rồi chạy lại ô 5–7 nếu cần.")

In [ ]:
# @title 6. Nạp model + LoRA, tự chọn GPU nhanh / CPU offload khi thiếu VRAM
import gc
import psutil
from diffusers import EulerAncestralDiscreteScheduler, StableDiffusionXLPipeline

requested_loras = {name for enabled, name in ((USE_ANATOMY_LORA, "anatomy"), (USE_EYE_LORA, "eyes")) if enabled}
if "lora_paths" not in globals():
    if requested_loras:
        raise RuntimeError("Hãy chạy ô 5 để tải/xác minh LoRA trước khi nạp model.")
elif set(lora_paths) != requested_loras or any(
    (lora_manifest[name]["weight"], lora_manifest[name]["manual"]) != (weight, manual)
    for name, weight, manual in
    (("anatomy", ANATOMY_WEIGHT, ANATOMY_LORA_PATH), ("eyes", EYE_WEIGHT, EYE_LORA_PATH))
    if name in requested_loras
):
    raise RuntimeError("Cấu hình LoRA đã đổi: chạy lại ô 5 để xác minh trước khi nạp model.")
if "pipe" in globals():
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
ram_gib = psutil.virtual_memory().available / 2**30
print(f"RAM hệ thống khả dụng: {ram_gib:.1f} GiB")
if ram_gib < 8:
    print("Cảnh báo: nạp checkpoint 6,94 GB có thể cần Colab high-RAM nếu phiên này quá ít RAM.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
free_bytes, _ = torch.cuda.mem_get_info()
# Mỗi LoRA thêm VRAM/RAM; chừa thêm vùng đệm trước khi chọn GPU trực tiếp.
auto_min_vram = (14 + 0.4 * len(globals().get("lora_paths", {}))) * 2**30
use_offload = VRAM_MODE == "low_vram" or (VRAM_MODE == "auto" and free_bytes < auto_min_vram)

def create_pipeline(offload):
    pipeline = StableDiffusionXLPipeline.from_single_file(
        str(checkpoint), torch_dtype=torch.float16, use_safetensors=True,
    )
    pipeline.scheduler = EulerAncestralDiscreteScheduler.from_config(pipeline.scheduler.config)
    pipeline.vae.enable_slicing()
    names = []
    weights = []
    for name, spec in globals().get("lora_manifest", {}).items():
        if name in lora_paths:
            path = inspect_lora(lora_paths[name], spec)  # kiểm lại ngay trước mỗi lần nạp / nạp lại khi OOM
            pipeline.load_lora_weights(
                str(path.parent), weight_name=path.name, adapter_name=name,
                local_files_only=True, use_safetensors=True,
            )
            names.append(name)
            weights.append(spec["weight"])
    if names:
        pipeline.set_adapters(names, adapter_weights=weights)
        print("LoRA đang dùng:", ", ".join(f"{name}={weight:.2f}" for name, weight in zip(names, weights)))
    if offload:
        pipeline.vae.enable_tiling()
        pipeline.enable_model_cpu_offload()
    else:
        pipeline.to("cuda")
    return pipeline

print("Chế độ:", "tiết kiệm VRAM (CPU offload)" if use_offload else "ưu tiên tốc độ (GPU FP16)")
gpu_oom = False
try:
    pipe = create_pipeline(use_offload)
except torch.cuda.OutOfMemoryError:
    gpu_oom = True
if gpu_oom:
    if VRAM_MODE != "auto" or use_offload:
        raise RuntimeError("Hết VRAM khi nạp model/LoRA. Tắt bớt LoRA hoặc chọn VRAM_MODE='low_vram', rồi chạy lại ô 5–6.")
    gc.collect()
    torch.cuda.empty_cache()
    print("Không đủ VRAM để nạp trực tiếp; đang tự thử CPU offload...")
    pipe = create_pipeline(True)
    use_offload = True
print("Model đã sẵn sàng. Chạy ô 7 để tạo ảnh; ô 8 sửa vùng lỗi nếu cần.")

In [ ]:
# @title 7. Chuẩn bị giao diện WAI bằng model đã xác minh
"""Personal WAI-illustrious Colab studio. Inlined into the standalone notebook.

The notebook's existing setup cells verify checkpoint and LoRA hashes before this
module is used. The model stays in Colab: Gradio only provides an authenticated UI.
"""

import gc
import json
import math
import os
import secrets
import threading
from datetime import datetime, timezone
from pathlib import Path

SIZE_PRESETS = (
    "512x512",
    "768x768",
    "768x1024",
    "1024x768",
    "832x1216",
    "1216x832",
    "1024x1024",
    "1024x1344",
    "1344x1024",
)
DEFAULT_PROMPT = (
    "general, 1girl, solo, cherry blossoms, spring, soft sunlight, "
    "detailed eyes, anime illustration, masterpiece, best quality"
)
DEFAULT_NEGATIVE = "nsfw, explicit, lowres, worst quality, bad anatomy, blurry"
REPAIR_HINTS = {
    "hands": (
        "natural hands, correct number of fingers, detailed fingers",
        "extra fingers, missing fingers, fused fingers, deformed hands",
    ),
    "legs": (
        "natural leg anatomy, well-formed feet, balanced pose",
        "extra legs, broken legs, deformed feet, extra toes",
    ),
    "eyes": (
        "symmetrical eyes, detailed irises, perfect eyes",
        "misaligned eyes, deformed eyes, extra eyes",
    ),
    "custom": ("", ""),
}


def _number(value, name, low, high, integer=False):
    if (
        isinstance(value, bool)
        or not isinstance(value, (float, int))
        or not math.isfinite(value)
    ):
        raise ValueError(f"{name} phải là số từ {low} đến {high}.")
    if value < low or value > high or (integer and int(value) != value):
        raise ValueError(f"{name} phải là số từ {low} đến {high}.")
    return int(value) if integer else float(value)


def _preset_size(size):
    if size not in SIZE_PRESETS:
        raise ValueError("Chọn kích thước có sẵn trong danh sách.")
    return tuple(map(int, size.split("x")))


def _normalize_source(image, mask=None):
    """Resize source and painted mask together, keeping their pixel alignment."""
    from PIL import Image

    if not isinstance(image, Image.Image):
        raise ValueError("Hãy tải ảnh nguồn PNG/JPG/WebP lên trước.")
    w, h = image.size
    if min(w, h) < 1 or w * h > 20_000_000 or max(w / h, h / w) > 1.75:
        raise ValueError(
            "Ảnh nguồn quá lớn hoặc quá dài (tối đa 20 MP và tỷ lệ 1,75:1). Hãy cắt ảnh trước."
        )
    if mask is not None and mask.size != image.size:
        raise ValueError(
            "Mask phải trùng chính xác kích thước ảnh nguồn trước khi thu nhỏ."
        )
    scale = min(1024 / max(w, h), max(1, 512 / min(w, h)))
    width = max(512, round(w * scale / 8) * 8)
    height = max(512, round(h * scale / 8) * 8)
    if width / height > 1.75:
        height = math.ceil(width / 1.75 / 8) * 8
    if height / width > 1.75:
        width = math.ceil(height / 1.75 / 8) * 8
    source = image.convert("RGB").resize((width, height), Image.Resampling.LANCZOS)
    if mask is None:
        return source, None
    return source, mask.convert("L").resize((width, height), Image.Resampling.NEAREST)


def _editor_mask(editor, uploaded_mask):
    """Editor background is source; transparent painted layers mark the repair area."""
    from PIL import Image, ImageChops

    if not isinstance(editor, dict) or not isinstance(
        editor.get("background"), Image.Image
    ):
        raise ValueError("Tải ảnh cần sửa vào khung vẽ trước.")
    original = editor["background"]
    if uploaded_mask is not None:
        if (
            not isinstance(uploaded_mask, Image.Image)
            or uploaded_mask.size != original.size
        ):
            raise ValueError(
                "Mask PNG tải lên phải cùng kích thước ảnh nguồn. Trắng = sửa, đen = giữ."
            )
        mask = uploaded_mask.convert("L")
    else:
        mask = Image.new("L", original.size, 0)
        for layer in editor.get("layers") or []:
            if not isinstance(layer, Image.Image) or layer.size != original.size:
                raise ValueError(
                    "Lớp tô không trùng kích thước ảnh nguồn. Hãy xóa lớp rồi vẽ lại."
                )
            # The editor's unpainted canvas is transparent. An opaque RGB layer
            # would select the entire image, so require a real alpha channel.
            if "A" not in layer.getbands():
                raise ValueError(
                    "Lớp tô thiếu kênh trong suốt. Dùng cọ trực tiếp trên ảnh, hoặc tải mask PNG."
                )
            mask = ImageChops.lighter(mask, layer.getchannel("A"))
    binary = mask.point(lambda value: 255 if value >= 64 else 0)
    if binary.getbbox() is None or binary.getextrema() == (255, 255):
        raise ValueError(
            "Hãy tô một vùng nhỏ để sửa (không để mask rỗng hoặc trắng toàn bộ)."
        )
    return _normalize_source(original, binary)


class StudioRuntime:
    def __init__(
        self,
        *,
        torch,
        pipe,
        create_pipeline,
        checkpoint,
        lora_paths,
        lora_manifest,
        vram_mode,
        use_offload,
        output_dir,
        drive_root,
        backup_dir="/content/wai_outputs",
    ):
        if pipe is None or not Path(checkpoint).is_file():
            raise RuntimeError(
                "Model chưa sẵn sàng; chạy lại các ô chuẩn bị checkpoint và nạp model."
            )
        self.torch = torch
        self.pipe = pipe
        self.create_pipeline = create_pipeline
        self.checkpoint = Path(checkpoint)
        self.lora_paths = dict(lora_paths)
        self.lora_manifest = dict(lora_manifest)
        self.vram_mode = vram_mode
        self.use_offload = bool(use_offload)
        self.output_dir = Path(output_dir)
        self.drive_root = Path(drive_root)
        self.backup_dir = Path(backup_dir)
        self.lock = threading.Lock()

    def _parameters(
        self,
        prompt,
        negative,
        steps,
        cfg,
        seed,
        count,
        anatomy_enabled,
        anatomy_weight,
        eyes_enabled,
        eyes_weight,
    ):
        if not isinstance(prompt, str) or not prompt.strip() or len(prompt) > 2000:
            raise ValueError("Prompt phải có từ 1 đến 2000 ký tự.")
        if not isinstance(negative, str) or len(negative) > 1500:
            raise ValueError("Negative prompt tối đa 1500 ký tự.")
        steps = _number(steps, "Steps", 10, 45, integer=True)
        cfg = _number(cfg, "CFG", 1, 12)
        seed = _number(seed, "Seed", -1, 2**32 - 1, integer=True)
        if seed != -1 and seed < 0:
            raise ValueError("Seed phải là -1 hoặc số từ 0 đến 2^32 - 1.")
        count = _number(count, "Số ảnh", 1, 4, integer=True)
        if not isinstance(anatomy_enabled, bool) or not isinstance(eyes_enabled, bool):
            raise ValueError("Công tắc LoRA không hợp lệ.")
        loras = {
            "anatomy": (anatomy_enabled, _number(anatomy_weight, "Anatomy", 0.1, 1)),
            "eyes": (eyes_enabled, _number(eyes_weight, "Eyes", 0.1, 1)),
        }
        for name, (enabled, _) in loras.items():
            if enabled and name not in self.lora_paths:
                raise ValueError(
                    f"LoRA {name} chưa được nạp. Bật nó ở ô cấu hình và chạy lại các ô tải/nạp model."
                )
        positive = prompt.strip()
        if loras["eyes"][0] and "perfect eyes" not in positive.lower():
            positive += ", perfect eyes"
        return positive, negative.strip(), steps, cfg, seed, count, loras

    def _apply_loras(self, choices):
        if self.lora_paths:
            names = list(self.lora_paths)
            # Weight zero disables a previously loaded adapter without deleting
            # the verified file or reloading the 6.94 GB checkpoint.
            weights = [choices[name][1] if choices[name][0] else 0.0 for name in names]
            # With CPU offload, the first inference may leave LoRA parameters as
            # inference tensors. PEFT's set_adapters toggles requires_grad; doing
            # that outside InferenceMode fails on the next image in PyTorch.
            with self.torch.inference_mode():
                self.pipe.set_adapters(names, adapter_weights=weights)

    def _infer_once(
        self,
        mode,
        source,
        mask,
        positive,
        negative,
        width,
        height,
        steps,
        cfg,
        seed,
        strength,
    ):
        generator = self.torch.Generator(device="cpu").manual_seed(seed)
        other_pipe = None
        if mode != "text":
            from diffusers import AutoPipelineForImage2Image, AutoPipelineForInpainting

            factory = (
                AutoPipelineForImage2Image
                if mode == "image"
                else AutoPipelineForInpainting
            )
            other_pipe = factory.from_pipe(self.pipe)  # reuse model weights
            if self.use_offload:
                self.pipe.remove_all_hooks()
                try:
                    other_pipe.enable_model_cpu_offload()
                except Exception:
                    self.pipe.enable_model_cpu_offload()
                    raise
        target = other_pipe or self.pipe
        options = dict(
            prompt=positive,
            negative_prompt=negative,
            width=width,
            height=height,
            num_inference_steps=steps,
            guidance_scale=cfg,
            generator=generator,
        )
        if mode != "text":
            options.update(image=source, strength=strength)
        if mode == "inpaint":
            options.update(mask_image=mask, padding_mask_crop=32)
        try:
            with self.torch.inference_mode():
                image = target(**options).images[0]
        finally:
            if other_pipe is not None and self.use_offload:
                other_pipe.remove_all_hooks()
                self.pipe.enable_model_cpu_offload()
        if image.size != (width, height):
            raise ValueError(
                "Pipeline trả về ảnh sai kích thước; không lưu để tránh lệch mask."
            )
        return image

    def _infer_with_retry(
        self,
        mode,
        source,
        mask,
        positive,
        negative,
        width,
        height,
        steps,
        cfg,
        seed,
        strength,
        choices,
    ):
        self._apply_loras(choices)
        oom = False
        try:
            return self._infer_once(
                mode,
                source,
                mask,
                positive,
                negative,
                width,
                height,
                steps,
                cfg,
                seed,
                strength,
            )
        except self.torch.cuda.OutOfMemoryError:
            oom = True
        if oom:
            self.torch.cuda.empty_cache()
            if self.vram_mode != "auto" or self.use_offload:
                raise RuntimeError(
                    "Hết VRAM. Giảm kích thước ảnh hoặc tắt LoRA và nạp lại model ở chế độ low_vram."
                )
            self.pipe = None
            gc.collect()
            self.torch.cuda.empty_cache()
            self.pipe = self.create_pipeline(True)  # same verified model + LoRAs
            self.use_offload = True
            self._apply_loras(choices)
            try:
                return self._infer_once(
                    mode,
                    source,
                    mask,
                    positive,
                    negative,
                    width,
                    height,
                    steps,
                    cfg,
                    seed,
                    strength,
                )
            except self.torch.cuda.OutOfMemoryError as exc:
                self.torch.cuda.empty_cache()
                raise RuntimeError(
                    "Vẫn thiếu VRAM sau khi thử CPU offload. Giảm kích thước ảnh."
                ) from exc

    def _save_png(self, image, mode, seed, metadata, embed):
        from PIL.PngImagePlugin import PngInfo

        filename = (
            f"wai_{mode}_{datetime.now(timezone.utc):%Y%m%d_%H%M%S_%f}_{seed}.png"
        )
        info = PngInfo()
        if embed:
            info.add_text("parameters", json.dumps(metadata, ensure_ascii=False))

        def save(directory):
            directory.mkdir(parents=True, exist_ok=True)
            path = directory / filename
            partial = directory / (filename + ".partial")
            try:
                image.save(partial, format="PNG", pnginfo=info)
                os.replace(partial, path)
            finally:
                partial.unlink(missing_ok=True)
            return path

        try:
            if (
                self.output_dir == self.drive_root
                or self.drive_root in self.output_dir.parents
            ) and not (self.drive_root / "MyDrive").is_dir():
                raise OSError("Google Drive đã ngắt kết nối")
            return save(self.output_dir)
        except OSError:
            if self.output_dir == self.backup_dir:
                raise
            return save(self.backup_dir)

    def _generate(
        self,
        mode,
        source,
        mask,
        target,
        feather,
        strength,
        size,
        prompt,
        negative,
        steps,
        cfg,
        seed,
        count,
        anatomy_enabled,
        anatomy_weight,
        eyes_enabled,
        eyes_weight,
        embed,
    ):
        from PIL import Image, ImageChops, ImageFilter

        positive, negative, steps, cfg, seed, count, choices = self._parameters(
            prompt,
            negative,
            steps,
            cfg,
            seed,
            count,
            anatomy_enabled,
            anatomy_weight,
            eyes_enabled,
            eyes_weight,
        )
        if mode == "text":
            width, height = _preset_size(size)
        elif mode == "image":
            width, height = _preset_size(size)
            if not isinstance(source, Image.Image):
                raise ValueError("Tải ảnh nguồn lên để dùng chế độ ảnh → ảnh.")
            if source.width * source.height > 20_000_000:
                raise ValueError("Ảnh nguồn tối đa 20 MP.")
            from PIL import ImageOps

            source = ImageOps.fit(
                ImageOps.exif_transpose(source).convert("RGB"),
                (width, height),
                Image.Resampling.LANCZOS,
            )
            strength = _number(strength, "Strength", 0.2, 0.85)
        elif mode == "inpaint":
            if target not in REPAIR_HINTS:
                raise ValueError("Chọn vùng sửa: tay, chân, mắt hoặc tùy chỉnh.")
            strength = _number(strength, "Strength", 0.2, 0.85)
            feather = _number(feather, "Làm mềm viền", 0, 24, integer=True)
            if source is None or mask is None:
                raise ValueError("Tải ảnh và tô hoặc tải mask PNG trước khi sửa.")
            if mask.getbbox() is None or mask.getextrema() == (255, 255):
                raise ValueError(
                    "Mask phải tô một vùng nhỏ, không để rỗng hoặc trắng toàn bộ."
                )
            width, height = source.size
            hint_pos, hint_neg = REPAIR_HINTS[target]
            if hint_pos:
                positive += ", " + hint_pos
            if hint_neg:
                negative = ", ".join(part for part in (negative, hint_neg) if part)
        else:
            raise ValueError("Chế độ tạo ảnh không được hỗ trợ.")
        if len(positive) > 2200 or len(negative) > 1700:
            raise ValueError("Prompt quá dài sau khi thêm gợi ý; hãy rút ngắn.")

        paths = []
        gallery = []
        selected = []
        with self.lock:  # one GPU pipeline, even if separate UI actions are clicked
            for index in range(count):
                image_seed = (
                    secrets.randbelow(2**32) if seed == -1 else (seed + index) % 2**32
                )
                image = self._infer_with_retry(
                    mode,
                    source,
                    mask,
                    positive,
                    negative,
                    width,
                    height,
                    steps,
                    cfg,
                    image_seed,
                    strength,
                    choices,
                )
                if mode == "inpaint":
                    blend = (
                        ImageChops.multiply(
                            mask, mask.filter(ImageFilter.GaussianBlur(radius=feather))
                        )
                        if feather
                        else mask
                    )
                    image = Image.composite(image.convert("RGB"), source, blend)
                metadata = {
                    "model": self.checkpoint.name,
                    "operation": mode,
                    "prompt": positive,
                    "negative_prompt": negative,
                    "seed": image_seed,
                    "width": width,
                    "height": height,
                    "steps": steps,
                    "cfg": cfg,
                    "strength": strength if mode != "text" else None,
                    "loras": [
                        {
                            "name": name,
                            "weight": choices[name][1],
                            "version": self.lora_manifest[name]["version"],
                            "sha256": self.lora_manifest[name]["sha256"],
                        }
                        for name in self.lora_paths
                        if choices[name][0]
                    ],
                }
                path = self._save_png(image, mode, image_seed, metadata, bool(embed))
                paths.append(str(path))
                gallery.append((str(path), f"Seed {image_seed} · {width}×{height}"))
                selected.append(str(image_seed))
        status = f"✅ Đã tạo {len(paths)} ảnh · seed: {', '.join(selected)} · đã lưu: {Path(paths[0]).parent}"
        return gallery, paths, status, paths[-1]

    def text_to_image(
        self,
        size,
        prompt,
        negative,
        steps,
        cfg,
        seed,
        count,
        anatomy_enabled,
        anatomy_weight,
        eyes_enabled,
        eyes_weight,
        embed,
    ):
        return self._generate(
            "text",
            None,
            None,
            None,
            0,
            1,
            size,
            prompt,
            negative,
            steps,
            cfg,
            seed,
            count,
            anatomy_enabled,
            anatomy_weight,
            eyes_enabled,
            eyes_weight,
            embed,
        )

    def image_to_image(
        self,
        source,
        size,
        strength,
        prompt,
        negative,
        steps,
        cfg,
        seed,
        count,
        anatomy_enabled,
        anatomy_weight,
        eyes_enabled,
        eyes_weight,
        embed,
    ):
        return self._generate(
            "image",
            source,
            None,
            None,
            0,
            strength,
            size,
            prompt,
            negative,
            steps,
            cfg,
            seed,
            count,
            anatomy_enabled,
            anatomy_weight,
            eyes_enabled,
            eyes_weight,
            embed,
        )

    def inpaint(
        self,
        editor,
        mask_file,
        target,
        strength,
        feather,
        prompt,
        negative,
        steps,
        cfg,
        seed,
        count,
        anatomy_enabled,
        anatomy_weight,
        eyes_enabled,
        eyes_weight,
        embed,
    ):
        source, mask = _editor_mask(editor, mask_file)
        return self._generate(
            "inpaint",
            source,
            mask,
            target,
            feather,
            strength,
            None,
            prompt,
            negative,
            steps,
            cfg,
            seed,
            count,
            anatomy_enabled,
            anatomy_weight,
            eyes_enabled,
            eyes_weight,
            embed,
        )


def build_app(runtime):
    """Build Gradio Blocks without opening a public tunnel until launch cell runs."""
    import gradio as gr
    from PIL import Image

    css = """
    .gradio-container {max-width: 1400px !important; margin: auto !important;}
    .studio-hero {padding: 25px 30px; border-radius: 18px; background: linear-gradient(118deg,#21182f,#382641 64%,#704065); color: #fff; margin-bottom: 16px; box-shadow: 0 12px 40px #140f202c;}
    .studio-hero h1 {color:#fff !important; font-size: 2rem; margin: 4px 0 8px;}
    .studio-hero p {color:#ecdce8; margin:0;}
    .studio-badge {font-size: 0.75rem; letter-spacing: 0.13rem; font-weight: bold; color:#f4b3dc;}
    .studio-notice {border-left: 3px solid #d891c2; padding: 9px 14px; background: #b088b61b; border-radius: 5px;}
    """
    with gr.Blocks(
        title="WAI Studio · Colab GPU",
        analytics_enabled=False,
        delete_cache=(3600, 3600),
    ) as demo:
        gr.HTML(
            "<div class='studio-hero'><span class='studio-badge'>✦ WAI · COLAB GPU · ANIME STUDIO</span><h1>Biến ý tưởng thành thế giới anime.</h1><p>WAI-illustrious v17 · LoRA tay/chân/mắt đã xác minh · Ảnh lưu ở Drive hoặc /content.</p></div>"
        )
        gr.Markdown(
            "**Không có đăng nhập:** bất kỳ ai biết URL tạm thời đều có thể dùng GPU Colab của bạn. Đừng chia sẻ link; dừng runtime để thu hồi. Model không chạy trên Cloudflare.",
            elem_classes="studio-notice",
        )
        with gr.Row():
            with gr.Column(scale=5, min_width=360):
                prompt = gr.Textbox(
                    label="Ý tưởng / prompt",
                    value=DEFAULT_PROMPT,
                    lines=3,
                    max_lines=6,
                    placeholder="Mô tả nhân vật, khung cảnh, ánh sáng, phong cách...",
                )
                negative = gr.Textbox(
                    label="Negative prompt", value=DEFAULT_NEGATIVE, lines=2
                )
                with gr.Accordion("⚙️ Thông số ảnh và LoRA", open=True):
                    with gr.Row():
                        steps = gr.Slider(
                            10, 45, value=25, step=1, label="Số bước (steps)"
                        )
                        cfg = gr.Slider(
                            1, 12, value=6, step=0.5, label="CFG / độ bám prompt"
                        )
                    with gr.Row():
                        seed = gr.Number(
                            value=-1,
                            minimum=-1,
                            maximum=2**32 - 1,
                            precision=0,
                            label="Seed (-1 = ngẫu nhiên)",
                        )
                        count = gr.Slider(
                            1, 4, value=1, step=1, label="Số ảnh (tạo lần lượt)"
                        )
                    anatomy = gr.Checkbox(
                        label="Anatomy Helper · tay/chân",
                        value="anatomy" in runtime.lora_paths,
                        interactive="anatomy" in runtime.lora_paths,
                    )
                    anatomy_weight = gr.Slider(
                        0.1,
                        1,
                        value=runtime.lora_manifest.get("anatomy", {}).get(
                            "weight", 0.55
                        ),
                        step=0.05,
                        label="Cường độ Anatomy",
                    )
                    eyes = gr.Checkbox(
                        label="Perfect Eyes · mắt",
                        value="eyes" in runtime.lora_paths,
                        interactive="eyes" in runtime.lora_paths,
                    )
                    eyes_weight = gr.Slider(
                        0.1,
                        1,
                        value=runtime.lora_manifest.get("eyes", {}).get("weight", 0.45),
                        step=0.05,
                        label="Cường độ Eyes",
                    )
                    embed = gr.Checkbox(
                        label="Nhúng prompt vào metadata PNG (tắt nếu chia sẻ ảnh)",
                        value=False,
                    )
                    gr.Markdown(
                        "LoRA chỉ có thể bật nếu đã chọn và xác minh ở ô cấu hình trước khi mở giao diện. Tắt/bật và đổi cường độ ở đây **không** tải lại checkpoint."
                    )
                shared = [
                    prompt,
                    negative,
                    steps,
                    cfg,
                    seed,
                    count,
                    anatomy,
                    anatomy_weight,
                    eyes,
                    eyes_weight,
                    embed,
                ]
                with gr.Tabs():
                    with gr.Tab("✦ Văn bản → ảnh"):
                        text_size = gr.Dropdown(
                            choices=list(SIZE_PRESETS),
                            value="1024x1024",
                            label="Kích thước",
                        )
                        text_button = gr.Button("Tạo ảnh từ prompt", variant="primary")
                    with gr.Tab("◈ Ảnh → ảnh"):
                        image_source = gr.Image(
                            label="Ảnh nguồn",
                            type="pil",
                            sources=["upload"],
                            image_mode="RGB",
                        )
                        image_size = gr.Dropdown(
                            choices=list(SIZE_PRESETS),
                            value="1024x1024",
                            label="Kích thước đầu ra",
                        )
                        image_strength = gr.Slider(
                            0.2, 0.85, value=0.45, step=0.05, label="Denoise strength"
                        )
                        gr.Markdown(
                            "Nếu ảnh nguồn có tỷ lệ khác kích thước đầu ra, giao diện sẽ cắt giữa ảnh để không làm méo."
                        )
                        image_button = gr.Button("Biến đổi ảnh", variant="primary")
                    with gr.Tab("✎ Sửa vùng ảnh"):
                        editor = gr.ImageEditor(
                            label="Tải ảnh vào đây và dùng cọ tô vùng cần sửa",
                            type="pil",
                            image_mode="RGBA",
                            sources=["upload"],
                            height=460,
                            format="png",
                            transforms=(),
                            brush=gr.Brush(
                                default_size=40, colors=["#ffffff"], color_mode="fixed"
                            ),
                            eraser=gr.Eraser(default_size=40),
                            layers=False,
                        )
                        mask_file = gr.Image(
                            label="Hoặc tải mask trắng/đen PNG (ưu tiên hơn vùng tô)",
                            type="pil",
                            image_mode="L",
                            sources=["upload"],
                        )
                        with gr.Row():
                            target = gr.Dropdown(
                                choices=["hands", "legs", "eyes", "custom"],
                                value="hands",
                                label="Chi tiết cần sửa",
                            )
                            inpaint_strength = gr.Slider(
                                0.2,
                                0.85,
                                value=0.45,
                                step=0.05,
                                label="Denoise strength",
                            )
                            feather = gr.Slider(
                                0,
                                24,
                                value=8,
                                step=2,
                                label="Làm mềm mép vùng sửa (px)",
                            )
                        gr.Markdown(
                            "**Vùng trắng / nét cọ = sửa; vùng đen = giữ nguyên.** Ảnh tải lên được thu về cạnh dài tối đa 1024 px cùng mask. Tránh tô toàn bộ ảnh."
                        )
                        inpaint_button = gr.Button("Sửa vùng đã tô", variant="primary")
            with gr.Column(scale=4, min_width=330):
                gallery = gr.Gallery(
                    label="Kết quả · nhấn để xem lớn",
                    columns=2,
                    height=550,
                    object_fit="contain",
                    format="png",
                    buttons=["download", "fullscreen"],
                )
                status = gr.Markdown(
                    "Ảnh đầu tiên sẽ xuất hiện tại đây. Model đã nạp trong Google Colab."
                )
                downloads = gr.File(
                    label="Tải ảnh PNG", file_count="multiple", interactive=False
                )
                latest = gr.State(None)
                with gr.Row():
                    to_image = gr.Button("Dùng ảnh mới nhất để biến đổi")
                    to_inpaint = gr.Button("Dùng ảnh mới nhất để sửa vùng")
                gr.Markdown(
                    "Ảnh được lưu tại Drive (nếu đã gắn) hoặc `/content/wai_outputs`. Tải xuống trước khi phiên Colab hết hạn nếu không dùng Drive."
                )

        outputs = [gallery, downloads, status, latest]
        events = (
            (text_button, runtime.text_to_image, [text_size, *shared]),
            (
                image_button,
                runtime.image_to_image,
                [image_source, image_size, image_strength, *shared],
            ),
            (
                inpaint_button,
                runtime.inpaint,
                [editor, mask_file, target, inpaint_strength, feather, *shared],
            ),
        )
        for button, fn, inputs in events:
            button.click(
                fn=fn,
                inputs=inputs,
                outputs=outputs,
                api_visibility="private",
                concurrency_id="wai_gpu",
                concurrency_limit=1,
            )

        def load_last(path):
            if not path or not Path(path).is_file():
                raise gr.Error("Hãy tạo ít nhất một ảnh trước.")
            return Image.open(path).convert("RGB")

        def edit_last(path):
            image = load_last(path).convert("RGBA")
            return {"background": image, "layers": [], "composite": image}

        to_image.click(
            fn=load_last,
            inputs=latest,
            outputs=image_source,
            api_visibility="private",
        )
        to_inpaint.click(
            fn=edit_last, inputs=latest, outputs=editor, api_visibility="private"
        )
        demo.queue(max_size=4, default_concurrency_limit=1, api_open=False)
    # Gradio 6 applies CSS and themes at launch, not in the Blocks constructor.
    demo.studio_theme = gr.themes.Soft(
        primary_hue="purple", secondary_hue="pink", neutral_hue="slate"
    )
    demo.studio_css = css
    return demo

if "pipe" not in globals():
    raise RuntimeError("Chưa có pipeline. Chạy các ô 1–6 theo thứ tự.")
studio_runtime = StudioRuntime(
    torch=torch, pipe=pipe, create_pipeline=create_pipeline,
    checkpoint=checkpoint, lora_paths=lora_paths, lora_manifest=lora_manifest,
    vram_mode=VRAM_MODE, use_offload=use_offload, output_dir=output_dir,
    drive_root=drive_root,
)
del pipe  # runtime owns the only reference, allowing OOM recovery to free VRAM
print("Đã chuẩn bị WAI Studio. Ô 8 sẽ tạo liên kết giao diện không cần đăng nhập.")


In [ ]:
# @title 8. Mở liên kết giao diện tạm thời (không cần đăng nhập)
from pathlib import Path

if "studio_runtime" not in globals():
    raise RuntimeError("Chưa nạp model. Chạy các ô 1–7 trước khi tạo link.")
# Đóng tunnel cũ khi cần tạo lại link, nhưng không nạp lại model.
if "studio_app" in globals():
    studio_app.close()
studio_app = build_app(studio_runtime)
allowed_outputs = [str(studio_runtime.output_dir.resolve()), str(studio_runtime.backup_dir.resolve())]
blocked_weights = [
    str(studio_runtime.checkpoint.resolve()),
    str((studio_runtime.drive_root / "MyDrive/AI/models").resolve()),
    str((studio_runtime.drive_root / "MyDrive/AI/loras").resolve()),
    "/content/wai_model_cache", "/content/wai_lora_cache",
    *(str(Path(path).resolve()) for path in studio_runtime.lora_paths.values()),
]
_, _, share_url = studio_app.launch(
    share=True, inline=False, prevent_thread_lock=True,
    server_name="127.0.0.1", max_file_size="12mb", footer_links=[],
    theme=studio_app.studio_theme, css=studio_app.studio_css,
    allowed_paths=allowed_outputs, blocked_paths=blocked_weights,
    enable_monitoring=False, show_error=True,
)
if not share_url:
    studio_app.close()
    raise RuntimeError("Gradio chưa tạo được link. Kiểm tra mạng Colab rồi chạy lại ô 8.")
print("Mở link:", share_url)
print("Không cần tài khoản/mật khẩu. Ai có link đều có thể dùng GPU Colab của bạn.")
print("Link ngừng hoạt động khi Colab dừng/ngắt. KHÔNG chia sẻ link; dừng runtime để thu hồi.")


**Khi gặp lỗi:** nếu ô 2 chỉ hiện dòng `ERROR: pip's dependency resolver...`, hãy xem ô đó có in `✅ Thư viện Studio đã sẵn sàng` không; riêng dòng này có thể là cảnh báo không chặn cài đặt. Nếu không có dấu ✅ hoặc có traceback, mở lại notebook mới nhất, chọn *Runtime → Restart runtime → Run all* và gửi đầy đủ traceback nếu vẫn lỗi (che link Gradio và thông tin riêng). Nếu không có GPU, chọn GPU trong Runtime; nếu OOM, chỉnh `VRAM_MODE=low_vram` hoặc tắt LoRA ở ô 3 rồi khởi động lại runtime. Nếu không tạo được URL, kiểm tra kết nối Colab/Gradio và chạy lại ô 8. **Link không có đăng nhập; đừng chia sẻ.** Notebook cơ sở và hash tài nguyên xem [README của dự án](https://github.com/manhlee1196-boop/ai-anime/tree/arena/01a0d84b-ai-anime).